### 01-tool-use-from-scratch

In [ ]:
import requests, json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "deepseek-r1"


def get_temperature(city: str) -> float:
    print("Fetching temperature for:", city)
    return 20.0


def main():
    user_input = input("Your question: ")

    prompt = f"""
                You are a helpful assistant. Answer the user's question in a friendly way.

                You can also use tools if you feel like they help you provide a better answer:
                - get_temperature(city: str) -> float: Get the current temperature for a given city.

                If you want to use one of these tools, you should output the tool name and its arguments in the following format:
                tool_name: arg1

                For example:
                get_temperature: New York

                With that in mind, answer the user's question:

                <user-question>
                {user_input}
                </user-question>

                If you request a tool, please output ONLY the tool call and nothing else.
            """

    payload = {
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False
                }

    response = requests.post(OLLAMA_URL, json=payload)
    response.raise_for_status()

    reply = response.json()["response"]

    print("Model reply:", reply)

    # TOOL CALL
    if reply.startswith("get_temperature:"):
        arg = reply.split(":", 1)[1].strip()

        temperature = get_temperature(arg)

        second_prompt = f"""
                            You are a helpful assistant. Answer the user's question in a friendly way.

                            User question:
                            {user_input}

                            Tool result:
                            The current temperature in {arg} is {temperature}°C.
                        """

        payload = {
                    "model": MODEL_NAME,
                    "prompt": second_prompt,
                    "stream": False
                    }

        response = requests.post(OLLAMA_URL, json=payload)
        response.raise_for_status()

        final_reply = response.json()["response"]

        print(final_reply)

    else:
        print(reply)


if __name__ == "__main__":
    main()

### 02-ollama-functions

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "deepseek-r1"


def get_temperature(city: str) -> float:
    print("Fetching temperature for:", city)
    return 20.0


def call_ollama(prompt: str, system: str | None = None, response_format=None) -> str:
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
    }

    if system:
        payload["system"] = system

    if response_format is not None:
        payload["format"] = response_format

    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["response"]


def main():
    user_input = input("Your question: ").strip()

    decision_schema = {
        "type": "object",
        "properties": {
            "action": {
                "type": "string",
                "enum": ["tool", "answer"]
            },
            "tool_name": {
                "type": "string"
            },
            "arguments": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string"
                    }
                },
                "additionalProperties": False
            },
            "answer": {
                "type": "string"
            }
        },
        "required": ["action"],
        "additionalProperties": False
    }

    system_prompt = (
                    "You are a helpful assistant. "
                    "You can answer normally, or decide to use a tool when needed. "
                    "Available tool: get_temperature(city: str) -> float. "
                    "If the user asks about current weather or temperature for a city, prefer the tool."
                    )

    first_prompt = f"""
                        Return ONLY valid JSON matching the provided schema.

                        Available tool:
                        - get_temperature(city: str) -> float

                        User question:
                        {user_input}
                    """.strip()

    raw_reply = call_ollama(prompt=first_prompt, system=system_prompt, response_format=decision_schema)

    try:
        decision = json.loads(raw_reply)
    except json.JSONDecodeError:
        print("Model returned invalid JSON:")
        print(raw_reply)
        return

    action = decision.get("action")

    if action == "tool":
        tool_name = decision.get("tool_name")
        arguments = decision.get("arguments", {})

        if tool_name != "get_temperature":
            print(f"Unknown tool requested by model: {tool_name}")
            return

        city = arguments.get("city")
        if not city:
            print("Tool call is missing required argument: city")
            return

        temperature = get_temperature(city)

        second_prompt = f"""
                            The user asked:
                            {user_input}

                            Tool used:
                            get_temperature(city="{city}")

                            Tool result:
                            The current temperature in {city} is {temperature}°C.

                            Now answer the user in a friendly way.
                        """.strip()

        final_reply = call_ollama(prompt=second_prompt, system="You are a helpful assistant. Answer clearly and briefly.")

        print(final_reply)

    elif action == "answer":
        print(decision.get("answer", "No answer returned by model."))

    else:
        print("Unexpected action from model:")
        print(decision)

if __name__ == "__main__":
    main()

### 03-multi-tool-versatile

In [ ]:
import json, re, sqlite3, requests
from datetime import datetime
from typing import List, Any

from dotenv import load_dotenv

from database import create_db_and_tables

load_dotenv()

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "deepseek-r1"
DB_FILE = "test_database.db"

create_db_and_tables()

def verify_customer(name: str, pin: str) -> int:
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    try:
        first_name, last_name = name.lower().split(maxsplit=1)
    except ValueError:
        conn.close()
        return -1

    cursor.execute(
                    """SELECT id FROM customers WHERE LOWER(first_name) = ? AND LOWER(last_name) = ? AND pin = ?""",(first_name, last_name, pin),
                    )
    result = cursor.fetchone()
    conn.close()

    if result:
        return result[0]
    return -1

def get_orders(customer_id: int) -> List[dict]:
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM orders WHERE customer_id = ?", (customer_id,))
    orders = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return orders

def check_refund_eligibility(customer_id: int, order_id: int) -> bool:
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT date FROM orders WHERE id = ? AND customer_id = ?",(order_id, customer_id))
    result = cursor.fetchone()
    conn.close()

    if not result:
        return False

    order_date = datetime.fromisoformat(result[0])
    return (datetime.now() - order_date).days <= 30

def issue_refund(customer_id: int, order_id: int) -> bool:
    print(f"Refund issued for order {order_id} for customer {customer_id}")
    return True


def share_feedback(customer_id: int, feedback: str) -> str:
    print(f"Feedback received from customer {customer_id}: {feedback}")
    return "Thank you for your feedback!"


available_functions = {
    "verify_customer": verify_customer,
    "get_orders": get_orders,
    "check_refund_eligibility": check_refund_eligibility,
    "issue_refund": issue_refund,
    "share_feedback": share_feedback,
}

TOOLS_DESCRIPTION = """
                        Available tools:
                        1. verify_customer(name: str, pin: str) -> int
                            - Verifies a customer's identity.
                            - Returns customer_id if verified, otherwise -1.

                        2. get_orders(customer_id: int) -> list[dict]
                            - Retrieves the order history for a verified customer.

                        3. check_refund_eligibility(customer_id: int, order_id: int) -> bool
                            - Checks if an order is eligible for a refund.

                        4. issue_refund(customer_id: int, order_id: int) -> bool
                            - Issues a refund for an order.
                            - This is a key action and MUST require confirmation first.

                        5. share_feedback(customer_id: int, feedback: str) -> str
                            - Stores customer feedback.
                    """.strip()

DECISION_SCHEMA = {
                    "type": "object",
                    "properties": {
                                "action": {
                                            "type": "string",
                                            "enum": ["respond", "tool"],
                                        },
                                "message": {
                                            "type": "string",
                                            },
                                "tool_name": {
                                            "type": "string",
                                            },
                                "arguments": {
                                            "type": "object",
                                            },
                                "requires_confirmation": {
                                                        "type": "boolean",
                                                        },
                                },
                    "required": ["action"],
                    "additionalProperties": False,
                    }


def call_ollama(prompt: str, system: str | None = None, response_format: dict | None = None) -> str:
    payload = {
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False,
                "options": {
                            "temperature": 0
                            }
                }

    if system:
        payload["system"] = system

    if response_format is not None:
        payload["format"] = response_format

    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["response"]


def extract_json(text: str) -> dict:
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return json.loads(match.group(0))

    raise ValueError(f"Model did not return valid JSON:\n{text}")

def serialize_tool_result(value: Any) -> str:
    if isinstance(value, (dict, list, bool, int, float)) or value is None:
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def execute_tool_call(tool_name: str, arguments: dict) -> str:
    if tool_name not in available_functions:
        return f"Unknown tool: {tool_name}"

    function_to_call = available_functions[tool_name]

    try:
        print(f"Calling {tool_name} with arguments: {arguments}")
        result = function_to_call(**arguments)
        return serialize_tool_result(result)
    except Exception as e:
        return f"Error calling {tool_name}: {e}"

def build_history_text(messages: list[dict]) -> str:
    lines = []
    for msg in messages:
        role = msg["role"].upper()
        content = msg["content"]
        lines.append(f"{role}: {content}")
    return "\n".join(lines)

def is_confirmation(text: str) -> bool:
    normalized = text.strip().lower()
    return normalized in {"yes", "y", "ano", "a", "confirm", "potvrdzujem", "suhlasim", "ok"}

def is_rejection(text: str) -> bool:
    normalized = text.strip().lower()
    return normalized in {"no", "n", "nie", "cancel", "zrusit", "nesuhlasim"}

def main():
    messages = []
    session_state = {
                    "verified_customer_id": None,
                    "pending_tool": None,
                    }

    system_prompt = """
                        You are a friendly and helpful customer service agent.

                        Rules you must follow:
                        - You must ALWAYS verify the customer's identity before providing any sensitive information.
                        - You must NOT expose any information to unverified customers.
                        - You must NOT provide any information that is not related to the customer's question.
                        - Do NOT guess any information.
                        - If you cannot perform a certain customer or order-related task, direct the user to a human agent.
                        - Ask for confirmation before performing any key actions.
                        - If the request is not related to customer service, say exactly:
                        "I'm sorry, I can't help with that."

                        Tool policy:
                        - Use tools when needed.
                        - NEVER call get_orders, check_refund_eligibility, issue_refund, or share_feedback unless the customer is already verified.
                        - Verification means session_state.verified_customer_id is not null, or you first call verify_customer successfully.
                        - issue_refund is a key action and MUST require confirmation before it is executed.
                        - If confirmation is needed, return action="respond" and ask for confirmation clearly.
                        - Keep answers concise and helpful.
                    """.strip()

    print("Welcome to the customer service chatbot! Type 'exit' to end the conversation.")

    while True:
        user_input = input("Your input: ").strip()
        if user_input.lower() == "exit":
            break

        if session_state["pending_tool"] is not None:
            pending = session_state["pending_tool"]

            if is_confirmation(user_input):
                tool_name = pending["tool_name"]
                arguments = pending["arguments"]

                tool_output = execute_tool_call(tool_name, arguments)
                messages.append({"role": "user", "content": user_input})
                messages.append({
                    "role": "tool",
                    "content": f"{tool_name}({json.dumps(arguments, ensure_ascii=False)}) -> {tool_output}"
                })

                session_state["pending_tool"] = None

            elif is_rejection(user_input):
                messages.append({"role": "user", "content": user_input})
                messages.append({"role": "assistant", "content": "Understood. I won't proceed with that action."})
                print("Understood. I won't proceed with that action.")
                session_state["pending_tool"] = None
                continue
            else:
                print("Please answer with yes or no.")
                continue
        else:
            messages.append({"role": "user", "content": user_input})

        for _ in range(5):
            history_text = build_history_text(messages)

            prompt = f"""
                        Return ONLY valid JSON matching the provided schema.

                        {TOOLS_DESCRIPTION}

                        Current session_state:
                        {json.dumps(session_state, ensure_ascii=False)}

                        Conversation so far:
                        <history>
                        {history_text}
                        </history>

                        Decide the next best step:
                        - If you can answer directly, return:
                        {{
                            "action": "respond",
                            "message": "your reply"
                        }}

                        - If you need a tool, return:
                        {{
                            "action": "tool",
                            "tool_name": "tool_name_here",
                            "arguments": {{ ... }},
                            "requires_confirmation": false
                        }}

                        Important:
                        - For issue_refund, never execute immediately without confirmation.
                        - Instead, ask for confirmation first with action="respond".
                        - If identity is not verified, do not reveal order information.
                    """.strip()

            raw_reply = call_ollama(prompt=prompt,system=system_prompt,response_format=DECISION_SCHEMA)

            try:
                decision = extract_json(raw_reply)
            except Exception as e:
                fallback_message = f"Model output error: {e}"
                print(fallback_message)
                messages.append({"role": "assistant", "content": fallback_message})
                break

            action = decision.get("action")

            if action == "respond":
                assistant_message = decision.get("message", "I'm sorry, I can't help with that.")
                print(assistant_message)
                messages.append({"role": "assistant", "content": assistant_message})
                break

            if action == "tool":
                tool_name = decision.get("tool_name")
                arguments = decision.get("arguments", {})
                requires_confirmation = decision.get("requires_confirmation", False)

                if not tool_name:
                    assistant_message = "I'm sorry, I can't help with that."
                    print(assistant_message)
                    messages.append({"role": "assistant", "content": assistant_message})
                    break

                if requires_confirmation:
                    session_state["pending_tool"] = {
                        "tool_name": tool_name,
                        "arguments": arguments,
                    }
                    assistant_message = decision.get(
                        "message",
                        f"Please confirm if you want me to proceed with {tool_name}."
                    )
                    print(assistant_message)
                    messages.append({"role": "assistant", "content": assistant_message})
                    break

                tool_output = execute_tool_call(tool_name, arguments)

                # aktualizacia session state po verify_customer
                if tool_name == "verify_customer":
                    try:
                        verified_customer_id = int(json.loads(tool_output))
                    except Exception:
                        try:
                            verified_customer_id = int(tool_output)
                        except Exception:
                            verified_customer_id = -1

                    if verified_customer_id > 0:
                        session_state["verified_customer_id"] = verified_customer_id

                messages.append({
                    "role": "tool",
                    "content": f"{tool_name}({json.dumps(arguments, ensure_ascii=False)}) -> {tool_output}"
                })
                continue

            assistant_message = "I'm sorry, I can't help with that."
            print(assistant_message)
            messages.append({"role": "assistant", "content": assistant_message})
            break

if __name__ == "__main__":
    main()

### 04-using-classes

In [ ]:
import json, re, sqlite3, requests
from datetime import datetime
from typing import Dict, Any
from dotenv import load_dotenv

from database import create_db_and_tables

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "deepseek-r1"
DB_FILE = "test_database.db"

create_db_and_tables()

class Tool:
    def __init__(
        self,
        name: str,
        description: str,
        parameters: Dict[str, Any],
    ):
        self.name = name
        self.description = description
        self.parameters = parameters

    def get_schema(self) -> Dict[str, Any]:
        return {
                "type": "function",
                "name": self.name,
                "description": self.description,
                "parameters": {
                                "type": "object",
                                "properties": self.parameters,
                                "additionalProperties": False,
                                "required": list(self.parameters.keys()),
                                }}

    def execute(self, arguments: Dict[str, Any]) -> Any:
        raise NotImplementedError("Each tool must implement its own execute method.")


class VerifyCustomerTool(Tool):
    def __init__(self):
        super().__init__(
            name="verify_customer",
            description="Verifies a customer's identity using their full name and PIN.",
            parameters={
                        "name": {
                                    "type": "string",
                                    "description": "The customer's full name, e.g., 'John Doe'.",
                                },
                        "pin": {
                                "type": "string",
                                "description": "The customer's PIN.",
                                },
                        })

    def execute(self, arguments: Dict[str, Any]) -> Any:
        try:
            full_name = str(arguments["name"]).strip().lower()
            pin = str(arguments["pin"]).strip()

            try:
                first_name, last_name = full_name.split(maxsplit=1)
            except ValueError:
                return -1

            conn = sqlite3.connect(DB_FILE)
            cursor = conn.cursor()
            cursor.execute("""SELECT id FROM customers WHERE LOWER(first_name) = ? AND LOWER(last_name) = ? AND pin = ?""",(first_name, last_name, pin))
            result = cursor.fetchone()
            conn.close()

            if result:
                return int(result[0])

            return -1
        except Exception as e:
            return f"Error in {self.name}: {e}"


class GetOrdersTool(Tool):
    def __init__(self):
        super().__init__(
            name="get_orders",
            description="Retrieves the order history for a verified customer.",
            parameters={
                        "customer_id": {
                                        "type": "integer",
                                        "description": "The customer's unique ID.",
                                        }
                        })

    def execute(self, arguments: Dict[str, Any]) -> Any:
        try:
            customer_id = int(arguments["customer_id"])

            conn = sqlite3.connect(DB_FILE)
            conn.row_factory = sqlite3.Row
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM orders WHERE customer_id = ?",(customer_id,))
            orders = [dict(row) for row in cursor.fetchall()]
            conn.close()

            return orders
        except Exception as e:
            return f"Error in {self.name}: {e}"


class CheckRefundEligibilityTool(Tool):
    def __init__(self):
        super().__init__(
            name="check_refund_eligibility",
            description="Checks if an order is eligible for a refund based on the order date.",
            parameters={
                "customer_id": {
                                "type": "integer",
                                "description": "The customer's unique ID.",
                                },
                "order_id": {
                            "type": "integer",
                            "description": "The unique ID of the order.",
                            },
                        })

    def execute(self, arguments: Dict[str, Any]) -> Any:
        try:
            customer_id = int(arguments["customer_id"])
            order_id = int(arguments["order_id"])

            conn = sqlite3.connect(DB_FILE)
            cursor = conn.cursor()
            cursor.execute("SELECT date FROM orders WHERE id = ? AND customer_id = ?",(order_id, customer_id))
            result = cursor.fetchone()
            conn.close()

            if not result:
                return False

            order_date = datetime.fromisoformat(result[0])
            return (datetime.now() - order_date).days <= 30
        except Exception as e:
            return f"Error in {self.name}: {e}"


class IssueRefundTool(Tool):
    def __init__(self):
        super().__init__(
            name="issue_refund",
            description="Issues a refund for an order.",
            parameters={
                "customer_id": {
                                "type": "integer",
                                "description": "The customer's unique ID.",
                                },
                "order_id": {
                            "type": "integer",
                            "description": "The unique ID of the order.",
                            },
                        })

    def execute(self, arguments: Dict[str, Any]) -> Any:
        try:
            customer_id = int(arguments["customer_id"])
            order_id = int(arguments["order_id"])

            # in reality, this would be stored in some database
            print(f"Refund issued for order {order_id} for customer {customer_id}")
            return True
        except Exception as e:
            return f"Error in {self.name}: {e}"


class ShareFeedbackTool(Tool):
    def __init__(self):
        super().__init__(
            name="share_feedback",
            description="Allows a customer to provide feedback about their experience.",
            parameters={
                "customer_id": {
                                "type": "integer",
                                "description": "The customer's unique ID.",
                                },
                "feedback": {
                                "type": "string",
                                "description": "The feedback text from the customer.",
                            },
                        })

    def execute(self, arguments: Dict[str, Any]) -> Any:
        try:
            customer_id = int(arguments["customer_id"])
            feedback = str(arguments["feedback"]).strip()
            print(f"Feedback received from customer {customer_id}: {feedback}")
            return "Thank you for your feedback!"
        except Exception as e:
            return f"Error in {self.name}: {e}"


class Agent:
    def __init__(self,model: str = MODEL_NAME,ollama_url: str = OLLAMA_URL):
        self.model = model
        self.ollama_url = ollama_url
        self.messages: list[Dict[str, str]] = []
        self.tools: Dict[str, Tool] = {}
        self.pending_action: Dict[str, Any] | None = None
        self.verified_customer_id: int | None = None
        self.system_prompt = ""

    def register_tool(self, tool: Tool):
        self.tools[tool.name] = tool

    def _get_tool_schemas(self) -> list[Dict[str, Any]]:
        return [tool.get_schema() for tool in self.tools.values()]

    def _serialize_tool_result(self, value: Any) -> str:
        if isinstance(value, (dict, list, bool, int, float)) or value is None:
            return json.dumps(value, ensure_ascii=False)
        return str(value)

    def _extract_json(self, text: str) -> Dict[str, Any]:
        text = text.strip()

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))

        raise ValueError(f"Model did not return valid JSON: {text}")

    def _build_history_text(self) -> str:
        lines = []
        for msg in self.messages:
            role = msg["role"].upper()
            content = msg["content"]
            lines.append(f"{role}: {content}")
        return "\n".join(lines)

    def _build_tools_text(self) -> str:
        blocks = []

        for tool in self.tools.values():
            schema = tool.get_schema()
            param_lines = []
            props = schema["parameters"]["properties"]

            for param_name, param_meta in props.items():
                param_type = param_meta.get("type", "string")
                param_desc = param_meta.get("description", "")
                param_lines.append(f'  - {param_name} ({param_type}): {param_desc}')

            block = (
                f"{schema['name']}: {schema['description']}\n"
                + "\n".join(param_lines)
            )
            blocks.append(block)

        return "\n\n".join(blocks)

    def _call_ollama(self, prompt: str, response_format: Dict[str, Any] | None = None) -> str:
        payload = {
                    "model": self.model,
                    "prompt": prompt,
                    "system": self.system_prompt,
                    "stream": False,
                    "think": False,
                    "options": {
                                "temperature": 0
                                },
                    }

        if response_format is not None:
            payload["format"] = response_format

        response = requests.post(self.ollama_url, json=payload, timeout=120)
        response.raise_for_status()
        data = response.json()
        return data["response"]

    def execute_tool_call(self, tool_name: str, arguments: Dict[str, Any]) -> str:
        if tool_name not in self.tools:
            return f"Unknown tool: {tool_name}"

        tool_to_call = self.tools[tool_name]

        try:
            print(f"Calling {tool_name} with arguments: {arguments}")
            result = tool_to_call.execute(arguments)
            return self._serialize_tool_result(result)
        except Exception as e:
            return f"Error calling {tool_name}: {e}"

    def run(self):
        raise NotImplementedError("The run method must be implemented by a subclass.")


class CustomerServiceAgent(Agent):
    def __init__(self, model: str = MODEL_NAME):
        super().__init__(model=model)
        self._set_initial_prompt()
        self._register_all_tools()

    def _set_initial_prompt(self):
        self.system_prompt = """
                                You are a friendly and helpful customer service agent.

                                Rules:
                                - You must ALWAYS verify the customer's identity before providing any sensitive information.
                                - You MUST NOT expose any information to unverified customers.
                                - You MUST NOT provide any information that is not related to the customer's question.
                                - DON'T guess any information.
                                - If you can't perform a certain customer or order-related task, direct the user to a human agent.
                                - Ask for confirmation before performing any key actions.
                                - If you can't help a customer or if a customer is asking for something that is not related to customer service, you MUST say:
                                "I'm sorry, I can't help with that."

                                Tool rules:
                                - Never use get_orders, check_refund_eligibility, issue_refund, or share_feedback before the customer is verified.
                                - Verification means verified_customer_id is not null.
                                - issue_refund is a key action and must always require confirmation.
                                - Keep answers concise and helpful.
                            """.strip()

    def _register_all_tools(self):
        tools_to_register = [
                                VerifyCustomerTool(),
                                GetOrdersTool(),
                                CheckRefundEligibilityTool(),
                                IssueRefundTool(),
                                ShareFeedbackTool(),
                            ]

        for tool in tools_to_register:
            self.register_tool(tool)

    def _decision_schema(self) -> Dict[str, Any]:
        return {
            "type": "object",
            "properties": {
                            "action": {
                                        "type": "string",
                                        "enum": ["respond", "tool", "confirm"],
                                        },
                            "message": {
                                        "type": "string",
                                        },
                            "tool_name": {
                                            "type": "string",
                                        },
                            "arguments": {
                                            "type": "object",
                                        },
                        },
                        "required": ["action"],
                        "additionalProperties": False,
                }

    def _needs_verification(self, tool_name: str) -> bool:
        return tool_name in {
                                "get_orders",
                                "check_refund_eligibility",
                                "issue_refund",
                                "share_feedback"
                            }

    def _is_key_action(self, tool_name: str) -> bool:
        return tool_name == "issue_refund"

    def _is_confirmation(self, text: str) -> bool:
        return text.strip().lower() in {
            "yes", "y", "ano", "a", "confirm", "potvrdzujem", "ok", "sure"
        }

    def _is_rejection(self, text: str) -> bool:
        return text.strip().lower() in {
            "no", "n", "nie", "cancel", "zrusit", "nesuhlasim"
        }

    def _inject_verified_customer_id(self, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        arguments = dict(arguments)

        if self._needs_verification(tool_name) and self.verified_customer_id is not None:
            arguments["customer_id"] = self.verified_customer_id

        return arguments

    def _append_tool_result_to_history(self, tool_name: str, arguments: Dict[str, Any], tool_output: str):
        self.messages.append(
            {
                "role": "tool",
                "content": f"{tool_name}({json.dumps(arguments, ensure_ascii=False)}) -> {tool_output}",
            }
        )

    def run(self):
        print("Welcome to the customer service chatbot! How can we help you today? Please type 'exit' to end the conversation.")

        while True:
            user_input = input("Your input: ").strip()
            if user_input.lower() == "exit":
                break

            if self.pending_action is not None:
                if self._is_confirmation(user_input):
                    self.messages.append({"role": "user", "content": user_input})

                    tool_name = self.pending_action["tool_name"]
                    arguments = self.pending_action["arguments"]

                    tool_output = self.execute_tool_call(tool_name, arguments)
                    self._append_tool_result_to_history(tool_name, arguments, tool_output)

                    self.pending_action = None

                elif self._is_rejection(user_input):
                    self.messages.append({"role": "user", "content": user_input})

                    assistant_message = "Understood. I won't proceed with that action."
                    self.messages.append({"role": "assistant", "content": assistant_message})
                    print(assistant_message)

                    self.pending_action = None
                    continue
                else:
                    print("Please answer with yes or no.")
                    continue
            else:
                self.messages.append({"role": "user", "content": user_input})

            for _ in range(5):  # limit tool call / assistant cycles to prevent infinite loops
                session_state = {
                    "verified_customer_id": self.verified_customer_id,
                    "pending_action": self.pending_action,
                }

                prompt = f"""
                            Return ONLY valid JSON matching this schema:
                            {json.dumps(self._decision_schema(), ensure_ascii=False)}

                            Available tools:
                            {self._build_tools_text()}

                            Current session state:
                            {json.dumps(session_state, ensure_ascii=False)}

                            Conversation so far:
                            <history>
                            {self._build_history_text()}
                            </history>

                            Choose exactly one next step:

                            1) Direct response:
                            {{
                            "action": "respond",
                            "message": "your reply"
                            }}

                            2) Tool call:
                            {{
                            "action": "tool",
                            "tool_name": "verify_customer",
                            "arguments": {{"name": "John Doe", "pin": "1234"}}
                            }}

                            3) Confirmation request for key action:
                            {{
                            "action": "confirm",
                            "message": "Please confirm that you want me to proceed with the refund.",
                            "tool_name": "issue_refund",
                            "arguments": {{"order_id": 123}}
                            }}

                            Important rules:
                            - Never reveal sensitive order information before verification.
                            - If the user is not verified, ask for full name and PIN.
                            - Do not ask the user for customer_id.
                            - For sensitive tools, Python will inject verified_customer_id automatically.
                            - issue_refund must always go through action='confirm' first.
                            - If the request is unrelated to customer service, respond exactly:
                            "I'm sorry, I can't help with that."
                            """.strip()

                raw_reply = self._call_ollama(prompt=prompt, response_format=self._decision_schema())

                try:
                    decision = self._extract_json(raw_reply)
                except Exception as e:
                    assistant_message = f"Model output error: {e}"
                    print(assistant_message)
                    self.messages.append({"role": "assistant", "content": assistant_message})
                    break

                action = decision.get("action")

                if action == "respond":
                    assistant_message = decision.get(
                        "message",
                        "I'm sorry, I can't help with that.",
                    )
                    print(assistant_message)
                    self.messages.append({"role": "assistant", "content": assistant_message})
                    break

                if action == "confirm":
                    tool_name = decision.get("tool_name", "")
                    arguments = decision.get("arguments", {})

                    if tool_name not in self.tools:
                        assistant_message = "I'm sorry, I can't help with that."
                        print(assistant_message)
                        self.messages.append({"role": "assistant", "content": assistant_message})
                        break

                    arguments = self._inject_verified_customer_id(tool_name, arguments)
                    self.pending_action = {"tool_name": tool_name,"arguments": arguments}

                    assistant_message = decision.get("message",f"Please confirm that you want me to proceed with {tool_name}.")
                    print(assistant_message)
                    self.messages.append({"role": "assistant", "content": assistant_message})
                    break

                if action == "tool":
                    tool_name = decision.get("tool_name", "")
                    arguments = decision.get("arguments", {})

                    if tool_name not in self.tools:
                        assistant_message = "I'm sorry, I can't help with that."
                        print(assistant_message)
                        self.messages.append({"role": "assistant", "content": assistant_message})
                        break

                    if self._needs_verification(tool_name) and self.verified_customer_id is None:
                        assistant_message = (
                                            "Before I can help with that, I need to verify your identity. "
                                            "Please provide your full name and PIN."
                                            )
                        print(assistant_message)
                        self.messages.append({"role": "assistant", "content": assistant_message})
                        break

                    arguments = self._inject_verified_customer_id(tool_name, arguments)

                    if self._is_key_action(tool_name):
                        self.pending_action = {"tool_name": tool_name,"arguments": arguments}
                        assistant_message = "Please confirm that you want me to proceed with the refund."
                        print(assistant_message)
                        self.messages.append({"role": "assistant", "content": assistant_message})
                        break

                    tool_output = self.execute_tool_call(tool_name, arguments)

                    if tool_name == "verify_customer":
                        try:
                            verified_id = json.loads(tool_output)
                            if isinstance(verified_id, int) and verified_id > 0:
                                self.verified_customer_id = verified_id
                        except Exception:
                            pass

                    self._append_tool_result_to_history(tool_name, arguments, tool_output)
                    continue

                assistant_message = "I'm sorry, I can't help with that."
                print(assistant_message)
                self.messages.append({"role": "assistant", "content": assistant_message})
                break


def main():
    agent = CustomerServiceAgent()
    agent.run()


if __name__ == "__main__":
    main()